## Member A Deliverable

This notebook builds a web scraper using Python and BeautifulSoup to collect raw web data related to three selected Sony products:

- Sony VAIO Laptops
- Sony PlayMemories Online
- Sony PlayStation Vue

The scraper extracts paragraph-level textual data from multiple relevant web sources and exports the results into a raw CSV file for downstream cleaning and analysis.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

In [14]:
!playwright install

In [2]:


headers = {
    "User-Agent": "Mozilla/5.0"
}

targets = [
    {
        "product": "Sony VAIO Laptops",
        "source": "Wikipedia",
        "url": "https://en.wikipedia.org/wiki/VAIO"
    },
    {
        "product": "Sony VAIO Laptops",
        "source": "Sony Global History",
        "url": "https://www.sony.com/en/SonyInfo/CorporateInfo/History/"
    },
    {
        "product": "Sony VAIO Laptops",
        "source": "Britannica Laptop Computer",
        "url": "https://www.britannica.com/technology/laptop-computer"
    },
    {
        "product": "Sony VAIO Laptops",
        "source": "Techradar",
        "url": "https://www.techradar.com/reviews/pc-mac/laptops-portable-pcs/laptops-and-netbooks/sony-vaio-e-series-984123/review"
    },
    {
        "product": "Sony VAIO Laptops",
        "source": "Techradar",
        "url": "https://www.techradar.com/news/tablets/laptops/mobile-computing/hands-on-sony-vaio-p-series-review-501155/2"
    },
    {
        "product": "Sony PlayMemories Online",
        "source": "Sony Support",
        "url": "https://support.d-imaging.sony.co.jp/www/disoft/int/playmemories-home/en/index.html"
    },
    {
        "product": "Sony PlayMemories Online",
        "source": "Sony Help Guide",
        "url": "https://helpguide.sony.net/di/pp/v1/en/contents/TP0000909109.html"
    },
    {
        "product": "Sony PlayMemories Online",
        "source": "Sony Product Page",
        "url": "https://www.sony.net/Products/di/en-us/products/playmemories-online/"
    },
    {
        "product": "Sony PlayMemories Online",
        "source": "DP Review",
        "url": "https://www.dpreview.com/forums/threads/playmemories-online-wow-just-wow.3990465/"
    },
    {
        "product": "Sony PlayStation Vue",
        "source": "Wikipedia",
        "url": "https://en.wikipedia.org/wiki/PlayStation_Vue"
    },
    {
        "product": "Sony PlayStation Vue",
        "source": "Britannica Streaming Media",
        "url": "https://www.britannica.com/topic/streaming-media"
    },
    {
        "product": "Sony PlayStation Vue",
        "source": "The Verge",
        "url": "https://www.theverge.com/2019/10/29/20938592/sony-playstation-vue-shutting-down-january-2020"
    }
]

rows = []
cutoff_date = datetime(2014, 2, 6) # Define the cutoff date for VAIO laptops

def scrape_targets():
    for target in targets:
        try:
            if target["source"] == "Techradar" and target["product"] == "Sony VAIO Laptops":
                print(f"Scraping Techradar article: {target['url']} with date filter...")
                response = requests.get(target["url"], headers=headers, timeout=15)
                soup = BeautifulSoup(response.text, "lxml")

                # Try to find the publication date within the article page
                published_date_str = None
                time_tag = soup.find('time', {'datetime': True})
                if time_tag:
                    published_date_str = time_tag['datetime']
                else:
                    # Also check meta tags, sometimes date is there
                    meta_tag = soup.find('meta', {'property': 'article:published_time'})
                    if meta_tag:
                        published_date_str = meta_tag['content']
                    else:
                        meta_tag = soup.find('meta', {'name': 'date'})
                        if meta_tag:
                            published_date_str = meta_tag['content']

                published_date = None
                if published_date_str:
                    try:
                        # Techradar uses ISO 8601 format with timezone usually
                        published_date = datetime.fromisoformat(published_date_str.replace('Z', '+00:00'))
                        # Convert to naive datetime for comparison if cutoff_date is naive
                        published_date = published_date.replace(tzinfo=None)
                    except ValueError as ve:
                        print(f"  - Could not parse date from '{published_date_str}' for {target['url']}. Error: {ve}")

                if published_date and published_date < cutoff_date:
                    print(f"  - Article date {published_date.strftime('%Y-%m-%d')} is before cutoff {cutoff_date.strftime('%Y-%m-%d')}. Scraping content.")
                    article_paragraphs = soup.find_all("p")
                    for i, p in enumerate(article_paragraphs, start=1):
                        text = p.get_text(" ", strip=True)
                        if len(text) > 50:
                            rows.append({
                                "product": target["product"],
                                "source": target["source"],
                                "url": target["url"],
                                "paragraph_id": i,
                                "raw_text": text
                            })
                elif published_date:
                    print(f"  - Skipping article: {target['url']} (Published: {published_date.strftime('%Y-%m-%d')} is after cutoff {cutoff_date.strftime('%Y-%m-%d')})")
                else:
                    print(f"  - No publish date found or parsed for {target['url']}. Skipping.")

                time.sleep(1) # Be respectful of the website

            else: # Existing logic for other sources
                print(f"Scraping {target['source']} for {target['product']}...")
                if target["source"] == "DP Review":
                    print(f"Starting DP Review scrape for {target['url']}")
                    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
                    print("Browser launched. Navigating to page...")
                    driver.get(target["url"])
                    print("Waiting for page to load...")
                    time.sleep(5)  # Wait for JS to load
                    print("Getting page content...")
                    content = driver.page_source
                    print(f"Content length: {len(content)}")
                    soup = BeautifulSoup(content, "lxml")
                    paragraphs = soup.find_all("div")
                    print(f"Found {len(paragraphs)} div elements.")
                    driver.quit()
                else:
                    response = requests.get(target["url"], headers=headers, timeout=15)
                    soup = BeautifulSoup(response.text, "lxml")
                    paragraphs = soup.find_all("p")

                if target["source"] == "DP Review": # Debugging for DP Review
                    print(f"  - Found {len(paragraphs)} potential content elements on {target['url']}.")

                for i, p in enumerate(paragraphs, start=1):
                    text = p.get_text(" ", strip=True)

                    if len(text) > 50:
                        rows.append({
                            "product": target["product"],
                            "source": target["source"],
                            "url": target["url"],
                            "paragraph_id": i,
                            "raw_text": text
                        })
                    elif target["source"] == "DP Review": # Debugging for DP Review
                        print(f"    - Skipping paragraph {i} from DP Review (length {len(text)} <= 50): {text[:50]}...")

                time.sleep(1)

        except Exception as e:
            print(f"Error scraping {target['url']}: {e}")
            rows.append({
                "product": target["product"],
                "source": target["source"],
                "url": target["url"],
                "paragraph_id": None,
                "raw_text": f"ERROR: {str(e)}"
            })

scrape_targets()

df = pd.DataFrame(rows)

df = df.drop_duplicates(subset=["product", "source", "url", "raw_text"]).reset_index(drop=True)

df.head(20)

Scraping Wikipedia for Sony VAIO Laptops...
Scraping Sony Global History for Sony VAIO Laptops...
Scraping Britannica Laptop Computer for Sony VAIO Laptops...
Scraping Techradar article: https://www.techradar.com/reviews/pc-mac/laptops-portable-pcs/laptops-and-netbooks/sony-vaio-e-series-984123/review with date filter...
  - Article date 2011-07-30 is before cutoff 2014-02-06. Scraping content.
Scraping Techradar article: https://www.techradar.com/news/tablets/laptops/mobile-computing/hands-on-sony-vaio-p-series-review-501155/2 with date filter...
  - Article date 2009-01-14 is before cutoff 2014-02-06. Scraping content.
Scraping Sony Support for Sony PlayMemories Online...
Scraping Sony Help Guide for Sony PlayMemories Online...
Scraping Sony Product Page for Sony PlayMemories Online...
Scraping DP Review for Sony PlayMemories Online...
Starting DP Review scrape for https://www.dpreview.com/forums/threads/playmemories-online-wow-just-wow.3990465/
Browser launched. Navigating to page..

,product,source,url,paragraph_id,raw_text
0,Sony VAIO Laptops,Wikipedia,https://en.wikipedia.org/wiki/VAIO,2,"VAIO Corporation ( VAIO 株式会社 , Baio Kabushiki ..."
1,Sony VAIO Laptops,Wikipedia,https://en.wikipedia.org/wiki/VAIO,3,"Vaio began as a brand of Sony , [ 5 ] introduc..."
2,Sony VAIO Laptops,Wikipedia,https://en.wikipedia.org/wiki/VAIO,4,"In 2025, JIP completed the sale of its 91.40% ..."
3,Sony VAIO Laptops,Wikipedia,https://en.wikipedia.org/wiki/VAIO,5,"Originally an acronym of ""Video Audio Input Ou..."
4,Sony VAIO Laptops,Wikipedia,https://en.wikipedia.org/wiki/VAIO,6,"As of 2023, Vaio is operational in the followi..."
5,Sony VAIO Laptops,Wikipedia,https://en.wikipedia.org/wiki/VAIO,7,"Although Sony made computers in the 1980s, suc..."
6,Sony VAIO Laptops,Wikipedia,https://en.wikipedia.org/wiki/VAIO,8,The PCV-90 was the first series of desktops in...
7,Sony VAIO Laptops,Wikipedia,https://en.wikipedia.org/wiki/VAIO,9,"Over the years, many audio visual technologies..."
8,Sony VAIO Laptops,Wikipedia,https://en.wikipedia.org/wiki/VAIO,10,"In 2001, Steve Jobs presented a VAIO PC runnin..."
9,Sony VAIO Laptops,Wikipedia,https://en.wikipedia.org/wiki/VAIO,11,Sony VAIO released later designs (2011 and lat...


### Debugging Techradar Scraping

In [3]:
print("Total rows:", len(df))
print("\nRows by product:")
print(df["product"].value_counts())

print("\nRows by source:")
print(df["source"].value_counts())

print("\nSample rows from all products:")
print(df.groupby("product").head(5))

Total rows: 363

Rows by product:
product
Sony PlayMemories Online    230
Sony VAIO Laptops            95
Sony PlayStation Vue         38
Name: count, dtype: int64

Rows by source:
source
DP Review                     216
Wikipedia                      64
Techradar                      44
Britannica Laptop Computer     12
Britannica Streaming Media     12
Sony Help Guide                 8
Sony Support                    6
The Verge                       1
Name: count, dtype: int64

Sample rows from all products:
                      product        source  \
0           Sony VAIO Laptops     Wikipedia   
1           Sony VAIO Laptops     Wikipedia   
2           Sony VAIO Laptops     Wikipedia   
3           Sony VAIO Laptops     Wikipedia   
4           Sony VAIO Laptops     Wikipedia   
95   Sony PlayMemories Online  Sony Support   
96   Sony PlayMemories Online  Sony Support   
97   Sony PlayMemories Online  Sony Support   
98   Sony PlayMemories Online  Sony Support   
99   Sony Pl

In [4]:
df.to_csv("sony_raw_data.csv", index=False)
print("sony_raw_data.csv saved successfully")

sony_raw_data.csv saved successfully


In [5]:
print("Scraping completed successfully.")
print("Output file: sony_raw_data.csv")
print("Number of rows collected:", len(df))
print("Columns:", list(df.columns))
print("\nRows by product:")
print(df["product"].value_counts())
print("\nRows by source:")
print(df["source"].value_counts())

Scraping completed successfully.
Output file: sony_raw_data.csv
Number of rows collected: 363
Columns: ['product', 'source', 'url', 'paragraph_id', 'raw_text']

Rows by product:
product
Sony PlayMemories Online    230
Sony VAIO Laptops            95
Sony PlayStation Vue         38
Name: count, dtype: int64

Rows by source:
source
DP Review                     216
Wikipedia                      64
Techradar                      44
Britannica Laptop Computer     12
Britannica Streaming Media     12
Sony Help Guide                 8
Sony Support                    6
The Verge                       1
Name: count, dtype: int64
